# J4-PM — Transformers & Hugging Face

Dans ce notebook on continue sur **SST-2** (la même tâche qu'en J4-AM) pour mesurer le gain apporté par les embeddings contextuels.

| Partie | Méthode | Attendu |
|--------|---------|----------|
| 1 | Explorer le tokenizer HuggingFace | sous-mots, padding, `attention_mask` |
| 2 | La `pipeline` API | sentiment, NER, QA — inférence clé en main |
| 3 | DistilBERT comme extracteur de features | token `[CLS]` + LogReg sklearn |
| 4 | Fine-tuning DistilBERT | `Trainer` end-to-end |
| 5 | Comparaison | GloVe LogReg → BERT frozen → DistilBERT fine-tuné |

In [1]:
!pip install -q transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00


In [ ]:
import logging
import warnings
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Silence les logs verbeux de HuggingFace
warnings.filterwarnings("ignore")
for logger_name in ("transformers", "datasets", "huggingface_hub"):
    logging.getLogger(logger_name).setLevel(logging.ERROR)

MODEL_NAME = "distilbert-base-uncased"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device : {DEVICE}")

## Partie 1 — Explorer le tokenizer HuggingFace

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Sous-mots : les mots rares sont découpés en fragments connus
examples = ["unhappiness", "tokenization", "DistilBERT", "cat"]
for word in examples:
    tokens = tokenizer.tokenize(word)
    print(f"{word!r:20s} → {tokens}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

'unhappiness'        → ['un', '##ha', '##pp', '##iness']
'tokenization'       → ['token', '##ization']
'DistilBERT'         → ['di', '##sti', '##lbert']
'cat'                → ['cat']


In [4]:
sentences = [
    "The movie was great!",
    "I absolutely hated this film.",
    "Ok.",
]

batch = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

print("input_ids      :", batch["input_ids"].shape)   # [3, L]
print("attention_mask :", batch["attention_mask"].shape)
print()
print("input_ids :\n", batch["input_ids"])
print("\nattention_mask (1=vrai token, 0=padding) :\n", batch["attention_mask"])

input_ids      : torch.Size([3, 8])
attention_mask : torch.Size([3, 8])

input_ids :
 tensor([[ 101, 1996, 3185, 2001, 2307,  999,  102,    0],
        [ 101, 1045, 7078, 6283, 2023, 2143, 1012,  102],
        [ 101, 7929, 1012,  102,    0,    0,    0,    0]])

attention_mask (1=vrai token, 0=padding) :
 tensor([[1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0]])


## Partie 2 — La `pipeline` API

La `pipeline` est l'interface qu'un dev utilisera **90 % du temps** : téléchargement du modèle, tokenisation, inférence et décodage — tout en une seule ligne. On explore ici les tâches les plus utiles en contexte applicatif.

In [ ]:
from transformers import pipeline

# Analyse de sentiment — modèle par défaut (distilbert fine-tuné sur SST-2)
sentiment = pipeline("sentiment-analysis")

phrases = [
    "This movie was absolutely fantastic!",
    "I completely wasted my time.",
    "It was neither good nor bad.",
    "One of the best films I've seen this year.",
    "Terrible acting, dreadful script.",
]
results = sentiment(phrases)
print(f"{'Sentiment':<10} {'Score':>6}  Phrase")
print("-" * 60)
for phrase, result in zip(phrases, results):
    icon = "✅" if result["label"] == "POSITIVE" else "❌"
    print(f"{icon} {result['label']:8s} {result['score']:>5.0%}  {phrase!r}")

In [ ]:
# Reconnaissance d'entités nommées (NER)
# Utile pour extraire automatiquement noms, lieux, organisations d'un texte
ner = pipeline("ner", aggregation_strategy="simple")

text = (
    "Hugging Face was founded in New York by Clément Delangue and Julien Chaumond in 2016. "
    "The company partners with Google, Microsoft and Amazon."
)
entities = ner(text)
print(f"Texte : {text}\n")
print(f"{'Type':<5}  {'Score':>5}  Entité")
print("-" * 40)
for e in entities:
    print(f"{e['entity_group']:<5}  {e['score']:>4.0%}  {e['word']!r}")

In [ ]:
# Question Answering extractif
# Le modèle localise la réponse dans un contexte fourni — idéal pour FAQ et chatbots
qa = pipeline("question-answering")

context = """
Hugging Face is an AI company that develops the Transformers library,
which provides thousands of pre-trained models for natural language processing,
computer vision, and audio tasks. The company was founded in 2016 and is
headquartered in New York City. Their Hub platform hosts over 900,000 models
and 200,000 datasets, used by researchers and developers worldwide.
The Trainer API simplifies fine-tuning of pre-trained models on custom datasets.
"""

questions = [
    "What does Hugging Face develop?",
    "When was Hugging Face founded?",
    "How many models are hosted on the Hub?",
    "What does the Trainer API do?",
]
for q in questions:
    answer = qa(question=q, context=context)
    print(f"Q : {q}")
    print(f"R : {answer['answer']}  ({answer['score']:.0%})\n")

### Utiliser son propre modèle fine-tuné dans une pipeline

Après le fine-tuning (Partie 4), on peut emballer le modèle dans une `pipeline` pour l'utiliser exactement comme n'importe quel modèle du Hub.

In [ ]:
# model_ft et tokenizer sont définis en Partie 4
model_ft.config.id2label = {0: "NEGATIVE", 1: "POSITIVE"}

ft_pipe = pipeline("sentiment-analysis", model=model_ft, tokenizer=tokenizer, device=0 if DEVICE == "cuda" else -1)

test_phrases = [
    "The acting was superb and the story truly gripping.",
    "Boring, predictable, and way too long.",
    "Not bad for a low-budget film.",
    "A masterpiece — I was moved to tears.",
    "Save your money and skip this one.",
]
results = ft_pipe(test_phrases)
print(f"{'Sentiment':<10} {'Score':>5}  Phrase")
print("-" * 65)
for phrase, result in zip(test_phrases, results):
    icon = "✅" if result["label"] == "POSITIVE" else "❌"
    print(f"{icon} {result['label']:8s} {result['score']:>4.0%}  {phrase!r}")

In [ ]:
# À vous ! Testez vos propres phrases
custom_phrases = [
    "Your sentence here...",
]
for phrase, result in zip(custom_phrases, ft_pipe(custom_phrases)):
    icon = "✅" if result["label"] == "POSITIVE" else "❌"
    print(f"{icon} {result['label']:8s} {result['score']:>4.0%}  {phrase!r}")

## Partie 3 — DistilBERT comme extracteur de features

On charge DistilBERT **gelé** (aucun paramètre mis à jour) et on utilise le vecteur du token spécial `[CLS]` — le premier token de chaque séquence — comme représentation de la phrase entière. On entraîne ensuite un simple LogReg par-dessus.

In [5]:
ds = load_dataset("glue", "sst2")

# Sous-ensemble pour ne pas attendre trop longtemps
N_TRAIN = 5000
train_texts  = ds["train"]["sentence"][:N_TRAIN]
train_labels = ds["train"]["label"][:N_TRAIN]
val_texts    = ds["validation"]["sentence"]
val_labels   = ds["validation"]["label"]

print(f"train : {len(train_texts)} exemples")
print(f"val   : {len(val_texts)} exemples")

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

train : 5000 exemples
val   : 872 exemples


In [6]:
bert = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert.eval()


def extract_cls(texts, tokenizer, model, batch_size=64):
    """Retourne un tableau (N, d) avec le vecteur [CLS] de chaque phrase."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(
            texts[i : i + batch_size],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls)
    return np.vstack(all_embeddings)


print("Extraction train…")
X_train = extract_cls(train_texts, tokenizer, bert)
print("Extraction val…")
X_val = extract_cls(val_texts, tokenizer, bert)
print(f"X_train : {X_train.shape}, X_val : {X_val.shape}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Extraction train…
Extraction val…
X_train : (5000, 768), X_val : (872, 768)


In [7]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_labels)
acc_frozen = accuracy_score(val_labels, clf.predict(X_val))
print(f"DistilBERT frozen + LogReg : {acc_frozen:.2%}")

DistilBERT frozen + LogReg : 82.11%


## Partie 4 — Fine-tuning DistilBERT

On laisse maintenant tous les paramètres se mettre à jour sur SST-2. Le `Trainer` gère la boucle d'entraînement, le padding dynamique, les checkpoints et l'évaluation.

In [8]:
raw_train = ds["train"].select(range(N_TRAIN))
raw_val   = ds["validation"]


def tokenize_dataset(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=128)


tok_train = raw_train.map(tokenize_dataset, batched=True)
tok_val   = raw_val.map(tokenize_dataset, batched=True)

# Le Trainer attend une colonne "labels" (et non "label")
tok_train = tok_train.rename_column("label", "labels")
tok_val   = tok_val.rename_column("label", "labels")
tok_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tok_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

In [9]:
model_ft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=DataCollatorWithPadding(tokenizer),
)

In [11]:
trainer.train()

{'eval_loss': '0.2711', 'eval_runtime': '1.479', 'eval_samples_per_second': '589.6', 'eval_steps_per_second': '9.466', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.3006', 'eval_runtime': '1.497', 'eval_samples_per_second': '582.5', 'eval_steps_per_second': '9.352', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.403', 'eval_runtime': '1.427', 'eval_samples_per_second': '610.8', 'eval_steps_per_second': '9.807', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '82.55', 'train_samples_per_second': '181.7', 'train_steps_per_second': '5.705', 'train_loss': '0.2002', 'epoch': '3'}


TrainOutput(global_step=471, training_loss=0.20020671609607218, metrics={'train_runtime': 82.5537, 'train_samples_per_second': 181.7, 'train_steps_per_second': 5.705, 'train_loss': 0.20020671609607218, 'epoch': 3.0})

In [12]:
preds_output = trainer.predict(tok_val)
y_pred = preds_output.predictions.argmax(axis=1)
acc_ft = accuracy_score(val_labels, y_pred)
print(f"DistilBERT fine-tuné : {acc_ft:.2%}")

DistilBERT fine-tuné : 89.11%


## Partie 5 — Comparaison

On résume les accuracies obtenues sur SST-2 tout au long des deux journées.

In [13]:
# Résultats J4-AM (issus du notebook j4am_embeddings_attention)
ACC_GLOVE_FROZEN  = 0.799   # GloVe frozen   + MLP
ACC_GLOVE_FT      = 0.836   # GloVe fine-tuné + MLP

results = {
    "GloVe frozen  (J4-AM)": ACC_GLOVE_FROZEN,
    "GloVe fine-tuné (J4-AM)": ACC_GLOVE_FT,
    "DistilBERT frozen (J4-PM)": acc_frozen,
    "DistilBERT fine-tuné (J4-PM)": acc_ft,
}

print(f"{'Méthode':<35} {'Accuracy':>9}")
print("-" * 46)
for name, acc in results.items():
    print(f"{name:<35} {acc:>8.2%}")

Méthode                              Accuracy
----------------------------------------------
GloVe frozen  (J4-AM)                 79.90%
GloVe fine-tuné (J4-AM)               83.60%
DistilBERT frozen (J4-PM)             82.11%
DistilBERT fine-tuné (J4-PM)          89.11%


### Lecture des résultats

**DistilBERT frozen ≈ GloVe fine-tuné** — c'est contre-intuitif, mais s'explique :
- Le token `[CLS]` d'un modèle **non fine-tuné** n'est pas optimisé pour la classification de sentiment ; c'est une représentation générique.
- GloVe fine-tuné, lui, a pu adapter ses embeddings directement sur la tâche.
- Avec seulement 5 000 exemples d'entraînement, le LogReg n'exploite pas pleinement la richesse des 768 dimensions de BERT.

**DistilBERT fine-tuné** (+6 points) — dès qu'on laisse tous les paramètres s'adapter, le modèle contextuel prend largement l'avantage : chaque token est représenté en fonction de son contexte, et la tête de classification est entraînée de bout en bout.